In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_platform_name", "gpu")
print(jax.local_devices()[0].device_kind)
from jax import numpy as np, random as jr, tree as jtu
import os


from zodiax.optimisation import sgd, adam

import amigo
import dorito


# visualisation
import matplotlib.pyplot as plt
import matplotlib as mpl
import ehtplot
import scienceplots
import cmasher as cmr

# matplotlib parameters
plt.style.use(["science", "bright", "no-latex"])

plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 8
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

inferno = mpl.colormaps["inferno"]
viridis = mpl.colormaps["viridis"]
seismic = mpl.colormaps["seismic"]
coolwarm = mpl.colormaps["coolwarm"]

inferno.set_bad("k", 0.5)
viridis.set_bad("k", 0.5)
seismic.set_bad("k", 0.5)
coolwarm.set_bad("k", 0.5)

In [ ]:
from socket import gethostname

if gethostname() == "glinton":
    morgana = "/media/morgana1/"
else:
    morgana = "/Volumes/morgana1/"

data_path = os.path.join(morgana, "snert/max/data/JWST/PDS70/calslope/")
uncal_path = os.path.join(morgana, "snert/max/data/JWST/PDS70/uncal/")
amigo_cache = os.path.join(morgana, "snert/max/data/amigo_files/")

cache = os.path.join(amigo_cache, "v_0.0.10/")
output_path = os.path.join(amigo_cache, "outputs/PDS70/")

EXP_TYPE = "NIS_AMI"
FILTERS = [
    "F480M",
    "F430M",
    "F380M",
    # "F277W",
]

# Bind file path, type and exposure type
file_fn = lambda data_path, filters=FILTERS, **kwargs: amigo.files.get_files(
    # "/Users/mcha5804/JWST/ERS1373/calgrps/",
    data_path,
    "calslope",
    EXP_TYPE=EXP_TYPE,
    FILTER=filters,
    **kwargs,
)

# Loading in data

In [ ]:
files = sorted(
    file_fn(data_path), key=lambda hdu: hdu[0].header.get("EXPMID", float("inf"))
)
# nsci = 1; ncal = 4
rolls = {}
sci_files = []
cal_files = []

for file in files:

    # manual bad pixel correction
    file["BADPIX"].data[58, 67] = 1
    file["BADPIX"].data[71, 22] = 1
    file["BADPIX"].data[65, 41] = 1
    file["BADPIX"].data[35, 70] = 1
    file["BADPIX"].data[70, 55] = 1
    file["BADPIX"].data[5, 5] = 1
    file["BADPIX"].data[-4, 37] = 1
    file["BADPIX"].data[51, 27] = 1
    file["BADPIX"].data[28, 18] = 1
    file["BADPIX"].data[32, 10] = 1

    file["BADPIX"].data[:, :3] = 1
    file["BADPIX"].data[:, -3:] = 1
    file["BADPIX"].data[:3, :] = 1
    file["BADPIX"].data[-3:, :] = 1

    if file[0].header["TARGPROP"] == "TD-PDS-70":
        file["BADPIX"].data[36:66, :25] = 1  # BACKGROUND STAR?

    if not bool(file[0].header["IS_PSF"]):
        sci_files.append(file)
    elif bool(file[0].header["IS_PSF"]):
        file[0].header["TARGPROP"] = "HD 228337"
        cal_files.append(file)
    else:
        print(f"Unkown target: {file[0].header['TARGPROP']}")

In [ ]:
from astropy.time import Time

for file in files:
    h = file[0].header
    t = Time(h["EXPMID"], format="mjd")
    print(
        f'{h["TARGPROP"]} {h["FILTER"]}, Dither {h["PATT_NUM"]}/{h["NUMDTHPT"]}, Roll {h["ROLL_REF"]:.1f}deg, {h["XPOSURE"] / 60:.1f}min, {t.iso}, Groups {h["NGROUPS"]}, Ints {h["NINTS"]}'
    )

In [ ]:
load_dict = lambda x: np.load(f"{x}", allow_pickle=True).item()

cal_exposures = [amigo.model_fits.PointFit(file) for file in cal_files]
sci_exposures = [amigo.model_fits.PointFit(file) for file in sci_files]
exposures = cal_exposures + sci_exposures
exposurse = exposures[:1]  # for testing


model = amigo.core_models.AmigoModel(
    exposures,
    optics=amigo.optical_models.AMIOptics(psf_upsample=1),
    detector=amigo.detector_models.LinearDetector(),
    ramp_model=amigo.ramp_models.NonLinearRamp(),
    read=amigo.read_models.ReadModel(),
    vis_model=None,
    state=load_dict(cache + "calibration.npy"),
)

In [ ]:
from scipy.ndimage import binary_dilation

exp = exposures[0]
print(exp.filter, exp.ngroups)

base_badpix = exp.badpix

threshold = 15_000

vmax = np.nanmax(np.where(base_badpix, np.nan, exp.ramp[-1]))
vmax2 = np.nanmax(np.where(base_badpix, np.nan, exp.slopes[0]))

badpixcube = []

for i, group in enumerate(exp.ramp[1:]):

    frame = np.where(base_badpix, np.nan, group)

    sat_badpix = binary_dilation(frame > threshold)
    badpix = sat_badpix | base_badpix
    slope = np.where(badpix, np.nan, exp.slopes[i])

    plt.figure(figsize=(8, 2))

    plt.subplot(131)
    plt.title(f"Group {i}")
    plt.imshow(frame, inferno, vmin=0, vmax=vmax)
    plt.axis("off")
    plt.colorbar()

    plt.subplot(132)
    plt.title("Saturation Bad Pixels")
    plt.imshow(sat_badpix, cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.colorbar()

    plt.subplot(133)
    plt.title("Combined Bad Pixels")
    plt.imshow(slope, inferno, vmin=0, vmax=vmax2)
    plt.axis("off")
    plt.colorbar()

    plt.tight_layout()
    plt.show()
    # plt.savefig(
    #     f"files/badpix/badpix_{exp.filter}_{i:02d}.png",
    #     dpi=300,
    #     bbox_inches="tight",
    # )
    # plt.close()

    badpixcube.append(badpix)

badpixcube = np.array(badpixcube)

In [ ]:
dorito.plotting.create_gif_from_dir("files/badpix/", "badpix.gif")

In [ ]:
def get_badpix_cube(exp, threshold=30_000):
    """
    Get a cube of dynamic bad pixels for an exposure.

    Parameters
    ----------
    exp : amigo.model_fits.PointFit
        The exposure to get the bad pixels for.
    threshold : float, optional
        The threshold for saturation, by default 30_000.

    Returns
    -------
    np.ndarray
        A cube of bad pixels.
    """
    base_badpix = exp.badpix

    for group in exp.ramp[1:]:
        group = np.where(base_badpix, np.nan, group)
        sat_badpix = binary_dilation(group > threshold)
        badpix = sat_badpix | base_badpix
        badpixcube.append(badpix)

    return np.array(badpixcube)

In [ ]:
from dLux import utils as dlu
from amigo.ramp_models import Ramp

x = exp.nuke_pixel_grads(model)
psf = exp.model_psf(x)
illuminance = exp.model_illuminance(psf, x)


# Get the charge (bias)
illum_small = dlu.downsample(illuminance.data, 3, mean=False)

# NOTE: This bias estimate is inadequate becuase it doesnt correctly account
# for the non-linear component of the gain. This ultimately should be properly
# calibrated, WITH the gain term using the ramp rather than slope data.
true_bias = model.read.gain * exp.ramp[0]
bias = true_bias - (illum_small / exp.ngroups)

# bias = self.ramp[0] - (illum_small / self.ngroups)
# bias = model.read.gain * bias

# Paste badpixels with median
bias = np.where(exp.badpix, np.median(bias), bias)

# Evolve the illuminance
ramp = model.ramp_model.evolve_illuminance(illuminance.data, bias, exp.ngroups)
ramp = Ramp(ramp, illuminance.pixel_scale)


ramp = exp.model_read(ramp, x)

In [ ]:
plt.imshow(ramp.data[0])